# Model Evaluation Notebook

Compare different models across 4 tasks before committing to the full pipeline.
All models accessed via OpenRouter.

1. **Transcription** — Gemini 2.5 Flash vs GPT Audio Mini vs Voxtral Small
2. **Audio Prosody** — Gemini 2.5 Flash vs GPT Audio Mini vs Voxtral Small
3. **OCR** — Gemini Flash vs Claude Sonnet
4. **Scene Description** — Gemini Flash (video clip input)

Estimated cost: ~$1-2 for all tests.

## Setup

In [3]:
import os
import base64
import time
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from tabulate import tabulate

load_dotenv("../.env")

openrouter = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

SAMPLES_DIR = Path("../data/samples")
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

# Models for audio tasks (transcription + prosody)
AUDIO_MODELS = {
    "Gemini 2.5 Flash": "google/gemini-2.5-flash",
    "Gemini 3 Flash": "google/gemini-3-flash-preview",
    "GPT Audio Mini": "openai/gpt-audio-mini",
    "Voxtral Small": "mistralai/voxtral-small-24b-2507",
}

print("Client initialized.")

Client initialized.


## Test Samples

Provide your own audio clips and keyframes directly.

Place files in `../data/samples/` or reference them by path below.

In [4]:
# Transcription test clips
transcription_clips = [
    "../data/samples/transcript1.wav",
    "../data/samples/transcript2.wav",
    "../data/samples/transcript3.wav",
]
transcription_labels = [
    "mumbling and background noise",
    "mumbling",
    "static interruptions",
]

# Prosody test clips
prosody_clips = [
    "../data/samples/prosody1.wav",
    "../data/samples/prosody2.wav",
    "../data/samples/prosody3.wav",
]
prosody_labels = [
    "shouting",
    "crying/distressed",
    "shouting commands",
]

# OCR test frames
ocr_frames = [
    "../data/samples/ocr1.png",
    "../data/samples/ocr2.png",
    "../data/samples/ocr3.png",
]
ocr_labels = [
    "simple license plate",
    "warped license plate",
    "far and blurry license plate",
]

# Scene description test clips (video input to Gemini Flash)
scene_clips = [
    "../data/samples/scene1.mp4",
    "../data/samples/scene2.mp4",
    "../data/samples/scene3.mp4",
]
scene_labels = [
    "lady in purple, man in blue, in the day",
    "smoke, topless man in red shorts, fire truck, at night",
    "many things on bed in hotel room",
]

# Verify all files exist
all_files = transcription_clips + prosody_clips + ocr_frames + scene_clips
for f in all_files:
    assert Path(f).exists(), f"File not found: {f}"

print(f"Ready: {len(transcription_clips)} transcription clips, {len(prosody_clips)} prosody clips, "
      f"{len(ocr_frames)} OCR frames, {len(scene_clips)} scene clips")

Ready: 3 transcription clips, 3 prosody clips, 3 OCR frames, 3 scene clips


## Helper Functions

In [12]:
import subprocess


def encode_file_b64(file_path: str) -> str:
    """Encode any file as base64."""
    with open(file_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def get_audio_format(audio_path: str) -> str:
    """Get the format string for OpenRouter audio input."""
    ext = Path(audio_path).suffix.lower()
    format_map = {".wav": "wav", ".mp3": "mp3", ".mp4": "mp4", ".m4a": "m4a"}
    return format_map.get(ext, ext.lstrip("."))


def get_image_media_type(image_path: str) -> str:
    """Get the MIME type for an image file."""
    ext = Path(image_path).suffix.lower()
    type_map = {".png": "image/png", ".jpg": "image/jpeg", ".jpeg": "image/jpeg"}
    return type_map.get(ext, "image/png")


def get_video_media_type(video_path: str) -> str:
    """Get the MIME type for a video file."""
    ext = Path(video_path).suffix.lower()
    type_map = {".mp4": "video/mp4", ".webm": "video/webm", ".mov": "video/mov"}
    return type_map.get(ext, "video/mp4")


def compress_video_720p(video_path: str) -> str:
    """Compress video to 720p if needed. Returns path to compressed file."""
    path = Path(video_path)
    
    # Check current resolution
    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "v:0",
         "-show_entries", "stream=height", "-of", "csv=p=0", str(path)],
        capture_output=True, text=True,
    )
    height = int(probe.stdout.strip())
    
    if height <= 720:
        print(f"  {path.name}: already {height}p, skipping compression")
        return video_path
    
    compressed_path = path.parent / f"{path.stem}_720p{path.suffix}"
    if compressed_path.exists():
        print(f"  {path.name}: 720p version already exists")
        return str(compressed_path)
    
    print(f"  {path.name}: compressing {height}p → 720p...")
    subprocess.run(
        ["ffmpeg", "-i", str(path), "-vf", "scale=-2:720",
         "-c:a", "copy", str(compressed_path), "-y", "-loglevel", "error"],
        check=True,
    )
    
    orig_mb = path.stat().st_size / (1024 * 1024)
    comp_mb = compressed_path.stat().st_size / (1024 * 1024)
    print(f"  {orig_mb:.1f}MB → {comp_mb:.1f}MB")
    return str(compressed_path)


def call_openrouter(model: str, messages: list, max_tokens: int = 2048) -> str:
    """Call a model via OpenRouter and return the text response."""
    start = time.time()
    response = openrouter.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=max_tokens,
    )
    elapsed = time.time() - start
    text = response.choices[0].message.content
    usage = response.usage
    print(f"  [{model}] {elapsed:.1f}s | {usage.prompt_tokens} in / {usage.completion_tokens} out")
    return text


OCR_PROMPT = (
    "Extract ALL visible text from this image. Include:\n"
    "- License plates (format: PLATE: XXX-XXXX)\n"
    "- Street signs, store signs, banners\n"
    "- Badge numbers, name tags\n"
    "- Screen text, timestamps, watermarks\n"
    "- Any other readable text\n\n"
    "For each piece of text, note its location and legibility "
    "(clear / partial / blurry). If no text is visible, say 'No text detected.'"
)

SCENE_PROMPT = """Analyze this body-worn camera footage segment. Return THREE separate sections:

=== SCENE DESCRIPTION ===
Describe everything visible in detail, in temporal order:
- EVENTS: What happens and when. Describe actions as they unfold (e.g., "officer approaches vehicle", "person exits car and walks away").
- PEOPLE: Every person visible — clothing colors/types, actions, position, approximate age/build.
- VEHICLES: Type, color, make/model if identifiable, notable features.
- OBJECTS: Everything visible, including small or background items.
- ENVIRONMENT: Indoor/outdoor, lighting, time of day, weather, location type.
- VISIBLE TEXT: Note any visible text, signs, license plates, or badges (just note their presence and content if readable — no need to be precise).
Be exhaustive — if a query like "person in a red shirt" or "vehicle pulled over at night" should match this footage, your description must contain those details.

=== TRANSCRIPT ===
Transcribe ALL speech verbatim with timestamps.
Format: [MM:SS] Speaker: "words spoken"
If multiple speakers, label them (Officer, Driver, Bystander, etc. or Speaker 1, Speaker 2).
If no speech is audible, write "No speech detected."

=== PROSODY ===
Analyze the audio characteristics:
{
  "has_shouting": true/false,
  "shouting_confidence": "high/medium/low",
  "max_volume": "quiet/normal/loud/very_loud",
  "emotional_tones": ["calm", "tense", "agitated", ...],
  "summary": "one sentence summary of audio characteristics"
}"""


def ocr_with_model(image_path: str, model_id: str) -> str:
    """Run OCR on an image using the specified model via OpenRouter."""
    img_b64 = encode_file_b64(image_path)
    media_type = get_image_media_type(image_path)
    messages = [{
        "role": "user",
        "content": [
            {
                "type": "image_url",
                "image_url": {"url": f"data:{media_type};base64,{img_b64}"},
            },
            {"type": "text", "text": OCR_PROMPT},
        ],
    }]
    return call_openrouter(model_id, messages)


def describe_scene_video(video_path: str, model_id: str) -> str:
    """Compress video to 720p if needed, then send via video_url format."""
    compressed_path = compress_video_720p(video_path)
    video_b64 = encode_file_b64(compressed_path)
    media_type = get_video_media_type(compressed_path)
    messages = [{
        "role": "user",
        "content": [
            {
                "type": "video_url",
                "video_url": {"url": f"data:{media_type};base64,{video_b64}"},
            },
            {"type": "text", "text": SCENE_PROMPT},
        ],
    }]
    return call_openrouter(model_id, messages)

---
## Test 1: Transcription

Compare 3 models (all via OpenRouter):
- **Gemini 2.5 Flash** — 3.1% WER on benchmarks, native audio input
- **GPT Audio Mini** — cost-efficient audio model from OpenAI
- **Voxtral Small 24B** — Mistral's audio-capable model

Key question: Which handles noisy body-cam audio best?

In [13]:
TRANSCRIPTION_PROMPT = (
    "Transcribe this audio clip verbatim. Include timestamps "
    "for each sentence or phrase. Format as:\n"
    "[MM:SS] text\n\n"
    "If you can distinguish multiple speakers, label them Speaker 1, Speaker 2, etc. "
    "Include non-speech sounds in brackets like [wind noise], [radio chatter]."
)


def transcribe_with_model(audio_path: str, model_id: str) -> str:
    """Transcribe audio using any model via OpenRouter."""
    audio_b64 = encode_file_b64(audio_path)
    audio_fmt = get_audio_format(audio_path)
    messages = [{
        "role": "user",
        "content": [
            {
                "type": "input_audio",
                "input_audio": {
                    "data": audio_b64,
                    "format": audio_fmt,
                },
            },
            {"type": "text", "text": TRANSCRIPTION_PROMPT},
        ],
    }]
    return call_openrouter(model_id, messages)

In [14]:
# Run transcription comparison across all models
transcription_results = []

for i, clip_path in enumerate(transcription_clips):
    print(f"\n{'='*60}")
    print(f"Transcription clip {i}: {transcription_labels[i]}")
    print(f"{'='*60}")

    results = {}
    for model_name, model_id in AUDIO_MODELS.items():
        print(f"\n--- {model_name} ---")
        results[model_name] = transcribe_with_model(clip_path, model_id)
        print(results[model_name])

    transcription_results.append(results)


Transcription clip 0: mumbling and background noise

--- Gemini 2.5 Flash ---


APIStatusError: Error code: 402 - {'error': {'message': 'Insufficient credits. This account never purchased credits. Make sure your key is on the correct account or org, and if so, purchase more at https://openrouter.ai/settings/credits', 'code': 402}}

### Transcription Notes

Write your observations here after running:
- Which model handled noisy audio better?
- Did either hallucinate words?
- Timestamp accuracy?
- Speaker separation quality?

---
## Test 2: Audio Prosody (Shouting / Raised Voice Detection)

Compare the same 3 models on detecting:
- Raised voices / shouting
- Emotional tone (calm, agitated, distressed)
- Background sounds (sirens, engines, radio)

Key concern: LLMs may exhibit "lexical dominance" — predicting emotion from
words rather than acoustic cues. Does the model detect shouting from tone,
or just from what's being said?

In [ ]:
PROSODY_PROMPT = """Analyze this audio clip from body-worn camera footage. Provide a structured assessment:

VOLUME:
- Overall volume level (quiet / normal / loud / very loud)
- Any volume spikes or sudden changes? Describe when and what.

VOICES:
- Number of distinct speakers
- Is anyone shouting or raising their voice? (yes/no, with confidence: high/medium/low)
- Emotional tone of each speaker (calm, tense, agitated, distressed, angry)

BACKGROUND SOUNDS:
- List all identifiable non-speech sounds (sirens, engines, wind, radio, footsteps, etc.)

STRUCTURED OUTPUT:
{
  "has_shouting": true/false,
  "shouting_confidence": "high/medium/low",
  "max_volume": "quiet/normal/loud/very_loud",
  "num_speakers": N,
  "emotional_tones": ["calm", ...],
  "background_sounds": ["wind", ...],
  "summary": "one sentence summary"
}"""


def analyze_prosody(audio_path: str, model_id: str) -> str:
    """Analyze audio prosody using any model via OpenRouter."""
    audio_b64 = encode_file_b64(audio_path)
    audio_fmt = get_audio_format(audio_path)
    messages = [{
        "role": "user",
        "content": [
            {
                "type": "input_audio",
                "input_audio": {
                    "data": audio_b64,
                    "format": audio_fmt,
                },
            },
            {"type": "text", "text": PROSODY_PROMPT},
        ],
    }]
    return call_openrouter(model_id, messages)

In [ ]:
# Run prosody analysis across all models
prosody_results = []

for i, clip_path in enumerate(prosody_clips):
    print(f"\n{'='*60}")
    print(f"Prosody clip {i}: {prosody_labels[i]}")
    print(f"{'='*60}")
    display(Audio(clip_path))

    results = {}
    for model_name, model_id in AUDIO_MODELS.items():
        print(f"\n--- {model_name} ---")
        results[model_name] = analyze_prosody(clip_path, model_id)
        print(results[model_name])

    prosody_results.append(results)

### Prosody Notes

Write your observations here:
- Did it correctly identify shouting vs normal speech?
- Were background sounds detected accurately?
- Is the structured JSON output reliable enough to filter on?

---
## Test 3: OCR (License Plates, Signs, Text)

Compare Gemini Flash vs Claude Sonnet on extracting text from frames.
This is where we hypothesized Sonnet would be more precise.

In [ ]:
# Models to compare for OCR
OCR_MODELS = {
    "Gemini 2.5 Flash": "google/gemini-2.5-flash-preview",
    "Gemini 3 Flash": "google/gemini-3-flash-preview",
    "GPT-4o-mini": "openai/gpt-4o-mini",
    "Qwen3-VL-32B": "qwen/qwen3-vl-32b-instruct",
}

ocr_results = []

for i, frame_path in enumerate(ocr_frames):
    print(f"\n{'='*60}")
    print(f"OCR frame {i}: {ocr_labels[i]}")
    print(f"{'='*60}")
    display(Image(filename=frame_path, width=400))

    results = {}
    for model_name, model_id in OCR_MODELS.items():
        print(f"\n--- {model_name} ---")
        results[model_name] = ocr_with_model(frame_path, model_id)
        print(results[model_name])

    ocr_results.append(results)

### OCR Notes

Write your observations:
- Did Sonnet catch text that Flash missed (or vice versa)?
- License plate accuracy difference?
- Did either model hallucinate text that isn't there?
- Is the cost difference justified?

---
## Test 4: Scene Description (Video Clip Quality Check)

Compare 4 models that accept native video input:
- **Gemini 2.5 Flash** — current plan baseline
- **Gemini 3 Flash** — newer Gemini generation
- **Qwen3-VL-32B** — top open-source VLM with native video support
- **NVIDIA Nemotron Nano 2 VL** — 12B model designed specifically for video understanding

Each model receives the full video clip (10-60s with audio) and must return
three structured sections: scene description, transcript, and prosody.

Key questions:
- Which produces the richest, most searchable descriptions?
- Do all models capture temporal events and audio cues?
- Is the cheaper/smaller model good enough?

In [ ]:
# Models to compare for scene description (all accept native video input)
VIDEO_MODELS = {
    "Gemini 2.5 Flash": "google/gemini-2.5-flash-preview",
    "Gemini 3 Flash": "google/gemini-3-flash-preview",
    "Qwen3-VL-32B": "qwen/qwen3-vl-32b-instruct",
    "Nemotron Nano 2 VL": "nvidia/nemotron-nano-12b-v2-vl",
}

scene_results = []

for i, clip_path in enumerate(scene_clips):
    print(f"\n{'='*60}")
    print(f"Scene {i}: {scene_labels[i]}")
    print(f"{'='*60}")

    results = {}
    for model_name, model_id in VIDEO_MODELS.items():
        print(f"\n--- {model_name} ---")
        results[model_name] = describe_scene_video(clip_path, model_id)
        print(results[model_name])

    scene_results.append(results)

In [ ]:
# Fill this in after running all tests
summary = [
    ["Transcription", "?", "?", "Which was more accurate on noisy audio?"],
    ["Prosody", "?", "?", "Which detected shouting/tone most reliably?"],
    ["OCR", "?", "?", "Did Sonnet catch more text?"],
    ["Scene Description", "Gemini Flash", "—", "Is the video output rich enough?"],
]

print(tabulate(
    summary,
    headers=["Task", "Winner", "Runner-up", "Notes"],
    tablefmt="github",
))

### Final Decision

Based on the tests above, update the model choices for the pipeline:

| Task | Planned Model | Actual Choice | Reasoning |
|------|--------------|---------------|-----------|
| Transcription | Gemini Flash | ? | |
| Prosody | Gemini Flash | ? | |
| OCR | Claude Sonnet | ? | |
| Scene Description | Gemini Flash (video) | ? | |